# 02 -- Metrics and Analysis

This notebook shows how to evaluate retrieval quality at every pipeline step: highlighting expected (relevant) documents, computing IR metrics, analyzing rank changes, and filtering to top-K documents.

In [ ]:
%matplotlib inline

import numpy as np

from rankflow import RankFlow

## Sample data

A 3-step pipeline with 8 documents. Two of them (`doc_a`, `doc_c`) are the ground-truth relevant documents.

In [ ]:
ranks = np.array([
    [3, 0, 5, 1, 2, 7, 4, 6],  # BM25
    [1, 2, 4, 0, 3, 7, 5, 6],  # Semantic
    [0, 3, 6, 1, 2, 7, 5, 4],  # Cross-Encoder
])
step_labels = ["BM25", "Semantic", "Cross-Encoder"]
chunk_labels = ["doc_a", "doc_b", "doc_c", "doc_d", "doc_e", "doc_f", "doc_g", "doc_h"]

## Highlighting relevant documents

Pass `relevant_chunks` to visually emphasize the ground-truth documents. Relevant docs are drawn with full opacity; irrelevant ones are faded.

In [ ]:
rf = RankFlow(
    ranks=ranks,
    step_labels=step_labels,
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c"],
)
rf.plot()

## Graded relevance

Instead of binary relevance, assign numeric grades. Higher grades get warmer colors via the relevance colormap.

In [ ]:
rf = RankFlow(
    ranks=ranks,
    step_labels=step_labels,
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c", "doc_d"],
    relevance_grades={"doc_a": 3, "doc_c": 2, "doc_d": 1},
)
rf.plot()

## Retrieval metrics per step

With `relevant_chunks` set, call `metrics(k)` to compute Precision@K, Recall@K, MRR, NDCG@K, and MAP at every step.

In [ ]:
rf = RankFlow(
    ranks=ranks,
    step_labels=step_labels,
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c"],
)

for step_label, m in zip(rf.step_labels, rf.metrics(k=3)):
    print(f"{step_label}: P@3={m['precision_at_k']:.2f}  R@3={m['recall_at_k']:.2f}  "
          f"MRR={m['mrr']:.2f}  NDCG@3={m['ndcg_at_k']:.3f}  MAP={m['map']:.3f}")

Use `metrics_df()` for a tidy pandas DataFrame (requires `pip install rankflow[pandas]`).

In [ ]:
rf.metrics_df(k=3)

## Showing metrics on the plot

Set `show_metrics=True` to annotate the plot with metric values at each step.

In [ ]:
rf = RankFlow(
    ranks=ranks,
    step_labels=step_labels,
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c"],
    show_metrics=True,
    top_k=5,
)
rf.plot()

## Rank delta annotations

Enable `show_deltas=True` to display the rank change between consecutive steps next to each line.

In [ ]:
rf = RankFlow(
    ranks=ranks,
    step_labels=step_labels,
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c"],
    show_deltas=True,
)
rf.plot()

## Per-chunk summary

The `summary()` method returns initial/final rank, max gain, max loss, and total displacement for each document.

In [ ]:
rf.summary_df()

## Top-K filtering

When you have many documents, focus the plot on the most important ones with `top_k`. Three modes are available:

- `"any"` (default): show documents that appear in top-K at **any** step
- `"initial"`: only those in top-K at the first step
- `"final"`: only those in top-K at the last step

In [ ]:
rf = RankFlow(
    ranks=ranks,
    step_labels=step_labels,
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c"],
    top_k=4,
    top_k_mode="any",
)
rf.plot()

In [ ]:
rf_final = RankFlow(
    ranks=ranks,
    step_labels=step_labels,
    chunk_labels=chunk_labels,
    relevant_chunks=["doc_a", "doc_c"],
    top_k=3,
    top_k_mode="final",
)
rf_final.plot()

---

**Next:** [03 -- Advanced Visualization](03_advanced_visualization.ipynb) covers A/B comparison, density plots, source provenance, and the interactive Plotly backend.